In [ ]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# 2. Load Dataset
data = fetch_california_housing()

df = pd.DataFrame(data.data, columns=data.feature_names)
df["MedHouseVal"] = data.target

print("First 5 rows:")
print(df.head())

# 3. Check Missing Values
print(df.isnull().sum())

# 4. Feature Scaling
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# 5. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# 6. Train Models

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# Random Forest
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# Gradient Boosting
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

# SVR
svr = SVR()
svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)


# 7. Model Evaluation
models = {
    "Linear Regression": y_pred_lr,
    "Decision Tree": y_pred_dt,
    "Random Forest": y_pred_rf,
    "Gradient Boosting": y_pred_gb,
    "SVR": y_pred_svr
}

results = []

for name, y_pred in models.items():
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "MSE": mse,
        "MAE": mae,
        "R2 Score": r2
    })

results_df = pd.DataFrame(results)

# 8. Display Results
print("\nModel Performance :\n")
print(results_df)

# 9. Best & Worst Model
best_model = results_df.loc[results_df["R2 Score"].idxmax()]
worst_model = results_df.loc[results_df["R2 Score"].idxmin()]

print("\nBest Performing Model:\n", best_model)
print("\nWorst Performing Model:\n", worst_model)
from sklearn.model_selection import cross_val_score, GridSearchCV

# Cross Validation (k=5)
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "SVR": SVR()
}

print("Cross Validation (R2 Scores):\n")

for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=5, scoring='r2')
    print(name, ":", scores.mean())
#K-fold cross-validation (k=5) was used to evaluate the model performance on different subsets of the dataset. 
#This ensures that the model is not overfitting and performs consistently.

# Hyperparameter Tuning

# Random Forest
rf_params = {
    "n_estimators": [50, 100],
    "max_depth": [None, 10]
}

rf_grid = GridSearchCV(RandomForestRegressor(), rf_params, cv=5)
rf_grid.fit(X_train, y_train)

print("\nBest RF Parameters:", rf_grid.best_params_)

# Gradient Boosting
gb_params = {
    "n_estimators": [50, 100],
    "learning_rate": [0.05, 0.1]
}

gb_grid = GridSearchCV(GradientBoostingRegressor(), gb_params, cv=5)
gb_grid.fit(X_train, y_train)

print("Best GB Parameters:", gb_grid.best_params_)